# Train Dreamer on DeepMind Control Walker

This notebook trains the high-level Dreamer agent on `walker-walk`, a DeepMind Control Suite task. The first run uses a short smoke-test budget; increase the step counts for a full experiment.


## Managed notebook install note

On Kaggle, Colab, and other managed notebook images, avoid a broad `pip install torchwm` in an already-initialized runtime if it starts replacing pinned CUDA, PyTorch, NumPy, or simulator packages. Use a fresh kernel and install TorchWM with only the DMC pieces needed for this notebook. If you have already changed MuJoCo or `dm_control` versions, restart the kernel before importing `torchwm`.


In [ ]:
# Optional managed-runtime install recipe. Leave commented unless you are setting up
# this notebook in a fresh Kaggle/Colab runtime.
#
# import sys, subprocess
# subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-deps", "torchwm"])
# subprocess.check_call([
#     sys.executable, "-m", "pip", "install",
#     "einops>=0.8.2",
#     "pyyaml>=6.0.3",
#     "tqdm>=4.67.1",
#     "requests>=2.32.0",
#     "click>=8.0.0",
#     "dm-control>=1.0.28",
#     "mujoco>=3.3.0",
#     "gymnasium>=1.2.2",
#     "opencv-python>=4.12.0.88",
#     "moviepy>=2.2.1",
# ])
#
# After running package-install cells, restart the notebook kernel before continuing.


In [ ]:
import importlib.metadata as metadata


def show_version(package_name: str) -> str:
    try:
        return metadata.version(package_name)
    except metadata.PackageNotFoundError:
        return "not installed"


for package_name in ["torchwm", "dm-control", "mujoco", "gymnasium", "torch", "torchvision"]:
    print(f"{package_name}: {show_version(package_name)}")


def check_dmc_walker() -> None:
    try:
        from dm_control import suite
    except ImportError as exc:
        raise ImportError(
            "DeepMind Control Suite is required for this notebook. Install the "
            "DMC extra with `pip install 'torchwm[dmc]'`. If `pip show "
            "dm-control mujoco` still reports missing packages, upgrade to a "
            "TorchWM release that includes the dmc extra or use the minimal "
            "managed-runtime recipe above, then restart the kernel."
        ) from exc

    try:
        env = suite.load("walker", "walk", task_kwargs={"random": 0})
        env.reset()
        env.physics.render(height=64, width=64, camera_id=0)
    except AttributeError as exc:
        message = str(exc)
        if "qLDiagSqrtInv" in message or "MjData" in message:
            raise RuntimeError(
                "The local dm-control and mujoco packages appear to be ABI-"
                "incompatible. Restart the kernel and reinstall a matching pair, "
                "for example with `pip install --upgrade 'dm-control>=1.0.28' 'mujoco>=3.3.0'`, "
                "then restart again before importing torchwm."
            ) from exc
        raise

    print("DMC walker smoke check passed.")


check_dmc_walker()


In [ ]:
import torch
import torchwm

print("Torch:", torch.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
cfg = torchwm.create_config(
    "dreamer",
    env_backend="dmc",
    env="walker-walk",
    total_steps=2_000,        # smoke test; use 1_000_000+ for research runs
    seed_steps=200,
    collect_steps=100,
    update_steps=10,
    batch_size=8,
    train_seq_len=16,
    checkpoint_interval=1_000,
    log_dir="runs/tutorial_dreamer_walker",
    enable_wandb=False,
)

agent = torchwm.create_model("dreamer", cfg)
agent


In [ ]:
# Long-running cell: uncomment when you are ready to train.
# agent.train()


## Next steps

- Switch `env` to another catalog entry such as `cartpole-swingup` or `cheetah-run`.
- Increase `total_steps`, `seed_steps`, and `batch_size` once the smoke run works.
- Resume by setting `cfg.restore = True` and `cfg.checkpoint_path` to a saved checkpoint.
